In [0]:
%pip install --upgrade yfinance pyarrow --quiet

In [0]:
import datetime
import time
import math
import warnings
import zoneinfo
from calendar import monthrange
 
import yfinance as yf
import pandas as pd
 
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, to_date
 
warnings.filterwarnings("ignore")
 
spark = SparkSession.builder.getOrCreate()

In [0]:
SYMBOL        = "^NSEI"
IST           = zoneinfo.ZoneInfo("Asia/Kolkata")
INTERVAL_SEC  = 60          # seconds between ticks
 
# Market session windows (HH, MM)
MARKET_OPEN   = (9,  15)
MARKET_CLOSE  = (15, 30)
PRE_MKT_END   = (10,  0)   # 09:15–10:00  → pre_market = 1
POST_MKT_ST   = (14, 30)   # 14:30–15:30  → post_market = 1
 
# Storage paths  — adjust to your DBFS mount
BASE_PATH     = "/Workspace/Users/hada.jai.hind@gmail.com/project_MarketMinds/Nifty_data_collection"
CSV_DIR       = f"{BASE_PATH}/csv"
PARQUET_DIR   = f"{BASE_PATH}/parquet"
DELTA_TABLE   = "nifty_live.session_ticks"
 
FLUSH_EVERY   = 5           # write to disk every N ticks (batching)
MAX_BAD_TICKS = 5           # abort after N consecutive fetch failures

In [0]:
# ── CELL 4 — Helper: expiry & calendar utils ──────────────────────────────────
 
def _last_thursday(year: int, month: int) -> datetime.date:
    """Last Thursday of a given month  (Nifty monthly expiry)."""
    last_day = monthrange(year, month)[1]
    d = datetime.date(year, month, last_day)
    # weekday(): Mon=0 … Thu=3 … Sun=6
    offset = (d.weekday() - 3) % 7
    return d - datetime.timedelta(days=offset)
 
 
def _next_thursday(ref: datetime.date) -> datetime.date:
    """Nearest Thursday on or after ref  (weekly expiry)."""
    days_ahead = (3 - ref.weekday()) % 7
    return ref + datetime.timedelta(days=days_ahead)
 
 
def _week_of_month(d: datetime.date) -> int:
    """1-indexed week number within the month."""
    first_weekday = d.replace(day=1).weekday()
    return (d.day + first_weekday - 1) // 7 + 1

In [0]:
def _session_flags(h: int, m: int) -> dict:
    """
    Returns a dict with:
      session       — 'pre' | 'regular' | 'post' | 'outside'
      pre_market    — 1 if 09:15–10:00
      post_market   — 1 if 14:30–15:30
      is_open_tick  — 1 if exactly 09:15
      is_close_tick — 1 if exactly 15:30
    """
    t       = h * 60 + m
    t_open  = MARKET_OPEN[0]  * 60 + MARKET_OPEN[1]
    t_pre   = PRE_MKT_END[0]  * 60 + PRE_MKT_END[1]
    t_post  = POST_MKT_ST[0]  * 60 + POST_MKT_ST[1]
    t_close = MARKET_CLOSE[0] * 60 + MARKET_CLOSE[1]
 
    if t_open <= t < t_pre:
        session, pre, post = "pre",     1, 0
    elif t_pre <= t < t_post:
        session, pre, post = "regular", 0, 0
    elif t_post <= t <= t_close:
        session, pre, post = "post",    0, 1
    else:
        session, pre, post = "outside", 0, 0
 
    return {
        "session"       : session,
        "pre_market"    : pre,
        "post_market"   : post,
        "is_open_tick"  : 1 if (h, m) == MARKET_OPEN  else 0,
        "is_close_tick" : 1 if (h, m) == MARKET_CLOSE else 0,
    }


In [0]:
def _data_quality(close: float, volume: float, prev_close: float | None) -> str:
    if volume == 0:
        return "ZERO_VOL"
    if prev_close and prev_close > 0:
        chg = abs(close - prev_close) / prev_close
        if chg > 0.05:         # >5% move in one 1-min bar → suspicious
            return "SPIKE"
    if close <= 0:
        return "BAD_PRICE"
    return "OK"

In [0]:
def is_market_open() -> bool:
    now = datetime.datetime.now(IST)
    if now.weekday() >= 5:                          # Sat / Sun
        return False
    t = datetime.time(now.hour, now.minute)
    return datetime.time(*MARKET_OPEN) <= t <= datetime.time(*MARKET_CLOSE)
 

In [0]:
# ── CELL 8 — Fetch + enrich one tick ─────────────────────────────────────────
 
def fetch_and_enrich(prev_close: float | None,
                     cum_tp_vol: float,
                     cum_vol: float) -> tuple[dict | None, float, float]:
    """
    Download latest 1-min bar from yfinance, attach all feature columns.
    Returns (row_dict | None, updated_cum_tp_vol, updated_cum_vol).
    """
    t0 = time.perf_counter()
    try:
        raw = yf.download(SYMBOL, period="1d", interval="1m", progress=False)
    except Exception as e:
        print(f"  [yfinance error] {e}")
        return None, cum_tp_vol, cum_vol
 
    latency_ms = (time.perf_counter() - t0) * 1000
 
    if raw.empty:
        return None, cum_tp_vol, cum_vol
 
    # ── Normalise column names (yfinance v0.2 returns MultiIndex) ─────────────
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw.columns = [c.lower() for c in raw.columns]
 
    # ── Timezone: always convert to IST ───────────────────────────────────────
    if raw.index.tz is None:
        raw.index = raw.index.tz_localize("UTC").tz_convert(IST)
    else:
        raw.index = raw.index.tz_convert(IST)
 
    # Remove duplicate bars, keep latest
    raw = raw[~raw.index.duplicated(keep="last")]
    bar = raw.iloc[-1]
 
    o  = float(bar.get("open",   0) or 0)
    hi = float(bar.get("high",   0) or 0)
    lo = float(bar.get("low",    0) or 0)
    cl = float(bar.get("close",  0) or 0)
    vo = float(bar.get("volume", 0) or 0)
 
    # ── Running intraday VWAP  (typical_price × volume) ───────────────────────
    tp          = (hi + lo + cl) / 3 if (hi + lo + cl) else cl
    cum_tp_vol += tp * vo
    cum_vol    += vo
    vwap        = cum_tp_vol / cum_vol if cum_vol else tp
 
    # ── Timestamp dims ────────────────────────────────────────────────────────
    now_ist = datetime.datetime.now(IST)
    d       = now_ist.date()
    h, m    = now_ist.hour, now_ist.minute
 
    # ── Expiry calculation ────────────────────────────────────────────────────
    is_expiry      = 1 if d.weekday() == 3 else 0        # Thursday = weekly expiry
    last_thu       = _last_thursday(d.year, d.month)
    is_monthly_exp = 1 if (is_expiry and d == last_thu) else 0
    next_thu       = _next_thursday(d)
    days_to_expiry = (next_thu - d).days
 
    # ── Session flags ─────────────────────────────────────────────────────────
    sess = _session_flags(h, m)
 
    # ── Additional ML-useful features ─────────────────────────────────────────
    price_range  = round(hi - lo, 2)
    return_pct   = round((cl - o) / o * 100, 4) if o else 0.0
    hl_ratio     = round(hi / lo, 4)             if lo else 1.0
    body_pct     = round(abs(cl - o) / (hi - lo) * 100, 2) if (hi - lo) > 0 else 0.0
    upper_shadow = round(hi - max(o, cl), 2)
    lower_shadow = round(min(o, cl) - lo, 2)
 
    row = {
        # ── Timestamp & date dims ─────────────────────────────────────────────
        "tick_ts"           : now_ist.replace(tzinfo=None),   # naive for Spark
        "tick_time_str"     : now_ist.strftime("%H:%M"),
        "date"              : d.day,
        "month"             : d.month,
        "year"              : d.year,
        "day"               : d.weekday(),         # 0=Mon … 6=Sun  ← your request
        "week_of_month"     : _week_of_month(d),
 
        # ── OHLCV ─────────────────────────────────────────────────────────────
        "open"              : round(o,  2),
        "high"              : round(hi, 2),
        "low"               : round(lo, 2),
        "close"             : round(cl, 2),
        "volume"            : round(vo, 0),
 
        # ── Derived price features (great for ML) ─────────────────────────────
        "typical_price"     : round(tp,          2),
        "price_range"       : price_range,           # high - low
        "return_pct"        : return_pct,            # (close-open)/open %
        "vwap"              : round(vwap,         2),# intraday running VWAP
        "hl_ratio"          : hl_ratio,              # high/low — volatility proxy
        "body_pct"          : body_pct,              # candle body % of range
        "upper_shadow"      : upper_shadow,          # wick above body
        "lower_shadow"      : lower_shadow,          # wick below body
 
        # ── Session / market flags ────────────────────────────────────────────
        "session"           : sess["session"],
        "pre_market"        : sess["pre_market"],    # 1 if 09:15–10:00
        "post_market"       : sess["post_market"],   # 1 if 14:30–15:30
        "is_open_tick"      : sess["is_open_tick"],  # 1 at exactly 09:15
        "is_close_tick"     : sess["is_close_tick"], # 1 at exactly 15:30
 
        # ── Expiry flags ──────────────────────────────────────────────────────
        "expiry"            : is_expiry,             # 1 = weekly Thursday expiry
        "is_monthly_expiry" : is_monthly_exp,        # 1 = last Thursday of month
        "days_to_expiry"    : days_to_expiry,        # calendar days to next expiry
 
        # ── Data quality & ops metadata ───────────────────────────────────────
        "data_quality"      : _data_quality(cl, vo, prev_close),
        "fetch_latency_ms"  : round(latency_ms, 1),
    }
    return row, cum_tp_vol, cum_vol


In [0]:
 
def bootstrap_delta_table():
    spark.sql("CREATE DATABASE IF NOT EXISTS nifty_live")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DELTA_TABLE} (
            tick_ts             TIMESTAMP,
            tick_time_str       STRING,
            date                INT,
            month               INT,
            year                INT,
            day                 INT,
            week_of_month       INT,
            open                DOUBLE,
            high                DOUBLE,
            low                 DOUBLE,
            close               DOUBLE,
            volume              DOUBLE,
            typical_price       DOUBLE,
            price_range         DOUBLE,
            return_pct          DOUBLE,
            vwap                DOUBLE,
            hl_ratio            DOUBLE,
            body_pct            DOUBLE,
            upper_shadow        DOUBLE,
            lower_shadow        DOUBLE,
            session             STRING,
            pre_market          INT,
            post_market         INT,
            is_open_tick        INT,
            is_close_tick       INT,
            expiry              INT,
            is_monthly_expiry   INT,
            days_to_expiry      INT,
            data_quality        STRING,
            fetch_latency_ms    DOUBLE
        )
        USING DELTA
        PARTITIONED BY (year, month, date)
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true',
            'delta.enableChangeDataFeed'        = 'true'
        )
    """)
    print(f"✅  Delta table ready → {DELTA_TABLE}")
 

In [0]:
def _ensure_dirs():
    import os
    os.makedirs(CSV_DIR,     exist_ok=True)
    os.makedirs(PARQUET_DIR, exist_ok=True)
 
 
def flush_to_delta(df: pd.DataFrame):
    sdf = spark.createDataFrame(df)
    sdf.write.format("delta") \
        .mode("append") \
        .saveAsTable(DELTA_TABLE)
 
 
def flush_to_csv(df: pd.DataFrame, session_date: str):
    import os
    path   = f"{CSV_DIR}/nifty_{session_date}.csv"
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", index=False, header=header)
 
 
def flush_to_parquet(df: pd.DataFrame, session_date: str):
    """Merge-append: re-read existing file + concat + overwrite."""
    path = f"{PARQUET_DIR}/nifty_{session_date}.parquet"
    try:
        existing = pd.read_parquet(path)
        df = pd.concat([existing, df], ignore_index=True)
    except FileNotFoundError:
        pass
    df.to_parquet(path, index=False, engine="pyarrow", compression="snappy")
 
 
def flush_all(buffer: list[dict], session_date: str):
    df = pd.DataFrame(buffer)
    df["tick_ts"] = pd.to_datetime(df["tick_ts"])
    flush_to_delta(df)
    flush_to_csv(df, session_date)
    flush_to_parquet(df, session_date)
    print(f"  💾  Flushed {len(buffer)} rows → Delta + CSV + Parquet")

In [0]:
 
def run():
    bootstrap_delta_table()
    _ensure_dirs()
 
    buffer       : list[dict] = []
    prev_close   : float | None = None
    cum_tp_vol   = 0.0
    cum_vol      = 0.0
    bad_streak   = 0
    session_date = datetime.datetime.now(IST).strftime("%Y_%m_%d")
 
    # ── Wait for market open ──────────────────────────────────────────────────
    print("⏳  Waiting for market open (09:15 IST) …")
    while True:
        now = datetime.datetime.now(IST)
        if now.weekday() >= 5:
            print("❌  Weekend — market closed.")
            return
        open_dt = now.replace(hour=9, minute=15, second=0, microsecond=0)
        close_dt= now.replace(hour=15,minute=30, second=0, microsecond=0)
        if now >= close_dt:
            print("❌  Past 15:30 — market closed for today.")
            return
        if now >= open_dt:
            break
        secs = (open_dt - now).total_seconds()
        print(f"  Opens in {int(secs//60)}m {int(secs%60)}s …")
        time.sleep(min(secs, 60))
 
    print("✅  Market OPEN — collecting every 60s …\n")
 
    # ── Tick loop ─────────────────────────────────────────────────────────────
    while is_market_open():
        now = datetime.datetime.now(IST)
 
        row, cum_tp_vol, cum_vol = fetch_and_enrich(prev_close, cum_tp_vol, cum_vol)
 
        if row is None:
            bad_streak += 1
            print(f"[{now.strftime('%H:%M:%S')}] ⚠  Fetch failed ({bad_streak}/{MAX_BAD_TICKS})")
            if bad_streak >= MAX_BAD_TICKS:
                print("💥  Too many failures — aborting.")
                break
            time.sleep(INTERVAL_SEC)
            continue
 
        bad_streak = 0
        prev_close = row["close"]
        buffer.append(row)
 
        q_icon = "✔" if row["data_quality"] == "OK" else "⚠"
        print(
            f"[{row['tick_time_str']}] {q_icon}  "
            f"O={row['open']:.2f}  H={row['high']:.2f}  "
            f"L={row['low']:.2f}  C={row['close']:.2f}  "
            f"VWAP={row['vwap']:.2f}  "
            f"sess={row['session']}  "
            f"exp={row['expiry']}  "
            f"Q={row['data_quality']}  "
            f"rows={len(buffer)}"
        )
 
        # ── Batch flush every FLUSH_EVERY ticks ───────────────────────────────
        if len(buffer) % FLUSH_EVERY == 0:
            flush_all(buffer[-FLUSH_EVERY:], session_date)
 
    # ── End-of-session: flush remainder ──────────────────────────────────────
    leftover = len(buffer) % FLUSH_EVERY
    if leftover:
        flush_all(buffer[-leftover:], session_date)
 
    print(f"\n🔔  Session complete — {len(buffer)} ticks collected.")
    print(f"📁  CSV     → {CSV_DIR}/nifty_{session_date}.csv")
    print(f"📁  Parquet → {PARQUET_DIR}/nifty_{session_date}.parquet")
    print(f"📁  Delta   → {DELTA_TABLE}")

In [0]:
run()